# CrewAI + Neo4j Client Showcase

This notebook demonstrates the CrewAI, Neo4j, NAMS, local MCP, and shared custom-tool integration in one guided workflow. Run the cells in order. Credentials stay in `../.env` and are never stored in this notebook.

## 1. Install dependencies

Run this once from a repository checkout. It installs CrewAI, the local MCP server, and the shared custom-tools package.

In [ ]:
%pip install -r ../requirements.txt
%pip install -e ..

## 2. Load local configuration

Create `crewai/.env` from `crewai/.env.example` first, then supply your OpenAI, Neo4j, and optional NAMS credentials. Do not place secrets in this notebook.

In [ ]:
from pathlib import Path
import os
import sys

from dotenv import load_dotenv

project_dir = Path.cwd().parent
sys.path.insert(0, str(project_dir))
load_dotenv(project_dir / '.env')

required = ('OPENAI_API_KEY', 'NEO4J_URI', 'NEO4J_USERNAME', 'NEO4J_PASSWORD')
missing = [setting for setting in required if not os.environ.get(setting)]
if missing:
    raise RuntimeError(f'Missing required settings in .env: {", ".join(missing)}')

print('Configuration loaded.')
print(f"Model: {os.environ.get('OPENAI_MODEL_NAME', 'default')}")
print(f"Local MCP: {'enabled' if os.environ.get('MCP_SERVER_COMMAND') else 'disabled'}")
print(f"NAMS: {'enabled' if os.environ.get('MEMORY_API_KEY') else 'disabled'}")

## 3. Verify local MCP and Neo4j access

This starts the official local Neo4j MCP server over stdio, lists its tools, and verifies that the configured graph is reachable.

In [ ]:
from agent.mcp import load_mcp_tools

mcp_tools = load_mcp_tools()
if not mcp_tools:
    raise RuntimeError('No local MCP tools were discovered. Set MCP_SERVER_COMMAND=neo4j-mcp-server.')

print('Discovered local MCP tools:')
for tool in mcp_tools:
    print(f'- {tool.name}: {tool.description}')

## 4. Inspect shared custom tools

The query agent always receives these tools. The LLM decides whether a question needs one. Some tools require a Companies/News graph schema; use local MCP `read-cypher` for general exploration of a NAMS graph.

In [ ]:
from agent.custom_tools import get_custom_tools

custom_tools = get_custom_tools()
for tool in custom_tools:
    print(f'- {tool.name}: {tool.description}')

## 5. Ask a natural-language graph question

Change `question` and rerun this cell. CrewAI chooses among local MCP, shared custom tools, built-in Neo4j tools, and NAMS tools. Set `CREW_VERBOSE=true` in `.env` to display tool execution details in the notebook output.

In [ ]:
from agent.crew import build_query_crew

question = 'Show all entities connected to DB and their relationship types.'
query_crew = build_query_crew(question)
query_result = query_crew.kickoff()
print(query_result)

## 6. Run the full company intelligence briefing

This demonstrates the specialized researcher, analyst, and writer agents. Use a company available in your graph.

In [ ]:
from agent.crew import build_company_intelligence_crew

company_name = 'Google'
briefing_crew = build_company_intelligence_crew(company_name)
briefing_result = briefing_crew.kickoff()
print(briefing_result)

## 7. Optional browser demo

For the browser UI, run the following in a terminal from `crewai/`, then open `http://127.0.0.1:8000/`:

```bash
uvicorn server:app --host 127.0.0.1 --port 8000
```